 # Walmart Weekly Sales Forecasting
 ### Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np

# Load datasets
train = pd.read_csv("./data/train.csv")
stores = pd.read_csv("./data/stores.csv")
features = pd.read_csv("./data/features.csv")

# Left join operations for creating the main table
data = pd.merge(train, stores, on="Store", how="left")
data = pd.merge(data, features, on=["Store", "Date"], how="left")

# Deleting duplicate IsHoliday and renaming the other one
data = data.drop(columns="IsHoliday_y")
data = data.rename(columns={"IsHoliday_x": "IsHoliday"})

# Fill NaN values in markdown features
markdown_columns = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
data[markdown_columns] = data[markdown_columns].fillna(0)

# Filter the data (only Store 1)
data = data[data['Store'] == 1].reset_index(drop=True)

# Collapse all departments into 1 row per week
feature_rules = {
    'Weekly_Sales': 'sum',
    'IsHoliday': 'max',
    'Temperature': 'mean',
    'Fuel_Price': 'mean',
    'CPI': 'mean',
    'Unemployment': 'mean',
    'MarkDown1': 'mean',
    'MarkDown2': 'mean',
    'MarkDown3': 'mean',
    'MarkDown4': 'mean',
    'MarkDown5': 'mean'
}
data = data.groupby('Date').agg(feature_rules).reset_index()

 ### Feature Engineering

In [ ]:
# Convert date (string to datetime)
data['Date'] = pd.to_datetime(data['Date'])
# Sort the data by Date
data = data.sort_values(by='Date').reset_index(drop=True)

# New features: Year, Month, Week of Year, Quarter of Year
data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month
data['Week_of_Year'] = data['Date'].dt.isocalendar().week.astype(int)
data['Quarter_of_Year'] = data['Date'].dt.quarter
# Convert IsHoliday (True/False to 1/0)
data['IsHoliday'] = data['IsHoliday'].astype(int)
# New Feature: Is_Pre_Holiday
data['Is_Pre_Holiday'] = data['IsHoliday'].shift(-1)
data['Is_Pre_Holiday'] = data['Is_Pre_Holiday'].fillna(0).astype(int)

# Lag Features: Lag 1, Lag 52
data['Sales_Lag_1'] = data['Weekly_Sales'].shift(1)
data['Sales_Lag_1'] = data['Sales_Lag_1'].fillna(0)
data['Sales_Lag_52'] = data['Weekly_Sales'].shift(52)
data['Sales_Lag_52'] = data['Sales_Lag_52'].fillna(0)

# New Feature: Sales Mean Last Month
data['Sales_Mean_Last_Month'] = data['Weekly_Sales'].shift(1).rolling(window=4, min_periods=1).mean()
data['Sales_Mean_Last_Month'] = data['Sales_Mean_Last_Month'].fillna(0)
data['Sales_Mean_Last_Month'] = data['Sales_Mean_Last_Month'].round(2)

 ### Data Splitting

In [ ]:
# Split data as train and test (05-02-2010 - 26-10-2012)
split_point=-4 # every 4 row equal to ONE month
train_data = data.iloc[:split_point].reset_index(drop=True)
test_data = data.iloc[split_point:].reset_index(drop=True)

# Target
y_train = train_data['Weekly_Sales']
# Train features
X_train = train_data.drop(columns=['Weekly_Sales', 'Date'], errors='ignore')
# Target validation
y_test = test_data['Weekly_Sales']
# Test
X_test = test_data.drop(columns=['Weekly_Sales', 'Date'], errors='ignore')

### Feature Selection

In [ ]:
import xgboost as xgb
from mrmr import mrmr_regression
from sklearn.feature_selection import RFE, SequentialFeatureSelector

# Create main model
main_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)

# Data have 18 features
feature_upper_limit = 12
feature_lower_limit = 6

# MRMR Selection
mrmr_cols = mrmr_regression(X=X_train, y=y_train, K=feature_upper_limit, show_progress=False)

# RFE Selection
rfe_sel = RFE(estimator=main_model, n_features_to_select=1).fit(X_train, y_train)
rfe_ranking = sorted(zip(rfe_sel.ranking_, X_train.columns))
rfe_cols = [col for rank, col in rfe_ranking]

# SFS Selection
sfs_sel = SequentialFeatureSelector(estimator=main_model, n_features_to_select=feature_upper_limit, direction='forward').fit(X_train, y_train)
sfs_cols = X_train.columns[sfs_sel.get_support()].tolist()

 ### Training and Validation

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Helper function
def train_and_evaluate(cols_to_use):
    main_model.fit(X_train[cols_to_use], y_train)
    preds = main_model.predict(X_test[cols_to_use])
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mape = mean_absolute_percentage_error(y_test, preds) * 100
    return preds, rmse, mape

# Result structures
all_results = []
best_predictions = {}
best_mape_tracker = {method: float('inf') for method in ["MRMR", "RFE", "SFS"]}
feature_counts = range(feature_lower_limit, feature_upper_limit + 1)

# Main loop
methods = {
    "MRMR": mrmr_cols,
    "RFE": rfe_cols,
    "SFS": sfs_cols
}

for k in feature_counts:
    for name, full_col_list in methods.items():
        # Slicing
        active_cols = full_col_list[:k]
        preds, rmse, mape = train_and_evaluate(active_cols)
        
        all_results.append({
            "Method": name, 
            "K": k, 
            "RMSE ($)": rmse, 
            "MAPE (%)": mape
        })
        
        # Store the best prediction
        if mape < best_mape_tracker[name]:
            best_mape_tracker[name] = mape
            best_predictions[name] = preds

# All features (apart from the loop)
preds_all, rmse_all, mape_all = train_and_evaluate(X_train.columns)
best_predictions["All Features"] = preds_all
all_results.append({
    "Method": "All Features",
    "K": len(X_train.columns),
    "RMSE ($)": rmse_all,
    "MAPE (%)": mape_all
})

# Creating the dataframe
exp_df = pd.DataFrame(all_results).round(2)
best_per_algorithm = (
    exp_df.loc[exp_df.groupby("Method")["MAPE (%)"].idxmin()]
    .sort_values(by="MAPE (%)")
    .reset_index(drop=True)
)

print(best_per_algorithm)

 ### Comperative Graphics of Feature Selection Algorithms

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style="whitegrid")

plot_df = pd.DataFrame({'Date': test_data['Date']})
for method in best_predictions.keys():
    plot_df[method] = best_predictions[method]

plot_df = plot_df.sort_values(by='Date').reset_index(drop=True)
actual_sales = y_test.values
date_strings = plot_df['Date'].dt.strftime('%Y-%m-%d').tolist()

methods_to_plot = [m for m in ["MRMR", "RFE", "SFS", "All Features"] if m in plot_df.columns]
num_methods = len(methods_to_plot)

fig, axes = plt.subplots(nrows=num_methods, ncols=1, figsize=(14, 4 * num_methods), sharex=True, sharey=True)

if num_methods == 1:
    axes = [axes]

y_min = actual_sales.min() * 0.85
y_max = actual_sales.max() * 1.15

for idx, method in enumerate(methods_to_plot):
    ax = axes[idx]
    
    ax.plot(date_strings, actual_sales, color='#111111', marker='o', linewidth=2.5, zorder=5, label='Actual Sales')
    
    if method != "All Features":
        k_val = int(best_per_algorithm.loc[best_per_algorithm['Method'] == method, 'K'].values[0])
        label_name = f"Predicted: {method} (K={k_val})"
    else:
        label_name = f"Predicted: {method}"
        
    color = sns.color_palette("Set2")[idx % 8]
    ax.plot(date_strings, plot_df[method], color=color, linestyle='--', marker='s', linewidth=2.0, label=label_name)
    
    ax.set_title(f"Method Evaluation: {method}", fontsize=12, fontweight='bold', pad=10)
    ax.set_ylabel("Weekly Sales ($)", fontsize=10)
    ax.set_ylim(y_min, y_max)
    ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: f"${int(x):,}"))
    ax.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='none', fontsize=9)

plt.xticks(rotation=30, ha='right')

fig.suptitle("Algorithm-Specific Test vs. Prediction Analysis", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

### Bayesian Optimization

In [ ]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer

# Get the best performed model and its feature count
best_method_row = best_per_algorithm.iloc[0]
best_method_name = best_method_row["Method"]
best_k_val = int(best_method_row["K"])

print(f"Best method: {best_method_name} (K={best_k_val})")

# Feature list of best model
if best_method_name == "MRMR":
    optimized_features = mrmr_cols[:best_k_val]
elif best_method_name == "RFE":
    optimized_features = rfe_cols[:best_k_val]
elif best_method_name == "SFS":
    optimized_features = sfs_cols[:best_k_val]
else:
    # if All features
    optimized_features = X_train.columns.tolist()

# Initialize search space
search_space = {
    'n_estimators': Integer(100, 250),
    'max_depth': Integer(3, 8),
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'subsample': Real(0.5, 1.0)
}

# Create base model
base_xgb = xgb.XGBRegressor(random_state=42, n_jobs=-1)

# Create bayes search
bayes_search = BayesSearchCV(
    estimator=base_xgb,
    search_spaces=search_space,
    n_iter=30,
    cv=3,
    scoring='neg_mean_absolute_percentage_error',
    random_state=42,
    n_jobs=-1
)

print(f"Bayesian Optimization starting...")
bayes_search.fit(X_train[optimized_features], y_train)

# print results
print("\nResults:")
print("Best Hyperparameters:", bayes_search.best_params_)

# Tabloyu dikey formatta oluştur ve display ile bastır
best_params_df = pd.DataFrame(list(bayes_search.best_params_.items()), columns=['Hyperparameter', 'Value'])
display(best_params_df) 

print("Best CV MAPE (%) Score:", round(-bayes_search.best_score_ * 100, 2))

# Validation with optimized model
tuned_model = bayes_search.best_estimator_
tuned_preds = tuned_model.predict(X_test[optimized_features])

tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_preds))
tuned_mape = mean_absolute_percentage_error(y_test, tuned_preds) * 100

print(f"Base Model MAPE (%): {round(best_method_row['MAPE (%)'], 2)}")
print(f"Tuned Model (Bayesian) MAPE (%): {round(tuned_mape, 2)}")